In [1]:
import os 
os.chdir('../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

!nvidia-smi

Tue Aug 19 05:15:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
| 32%   48C    P8             39W /  200W |      11MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os
import math
import numpy as np
from easydict import EasyDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

In [3]:
# ===============================
# Config
# ===============================
from torchvision.models import (
    vit_b_16, ViT_B_16_Weights,
    vit_l_16, ViT_L_16_Weights,
    vit_h_14, ViT_H_14_Weights,
)

config = EasyDict()
config.backbone      = 'DiT'
config.train_pt_dir  = '/dataset/train4.0_10k'
config.valid_pt_dir  = '/dataset/eval4.0'
config.batch_size    = 10
config.CFG           = 4.0
config.val_every     = 100
config.log_dir       = "logs/CFG4.0/0818-16:Taylor,ViT-B,GAP"
config.latent_size = (4, 32, 32)

# LR & Scheduler
config.base_lr       = 2e-3
config.total_steps   = 100*1000        # 전체 학습 스텝

# Loss
config.classifier = EasyDict()
config.classifier.label_smoothing = 0.0
# for ViT-H
# config.classifier.arch = "vit_h_14"
# config.classifier.weights = ViT_H_14_Weights.IMAGENET1K_SWAG_E2E_V1
# config.classifier.image_size = (518, 518)
# for ViT-B
config.classifier.arch = "vit_b_16"
config.classifier.weights = ViT_B_16_Weights.IMAGENET1K_V1
config.classifier.image_size = (224, 224)

config.losses = ['inception', 'PSNR', 'classifier']
config.main_loss = 'classifier'

os.makedirs(config.log_dir, exist_ok=True)


In [4]:
# ===============================
# Model (frozen)
# ===============================
from backbones.dit import DiT
from utils.inception import FIDInception
from utils.vit import ViTClassifier

if config.backbone == 'DiT':
    model = DiT(trainable=True)  # 내부 구현에 맞춰 유지
    model.set_freeze()
device = model.device
print(model)
inception = FIDInception().to(device)
if 'classifier' in config.losses:
    classifier = ViTClassifier(arch=config.classifier.arch,
                            weights=config.classifier.weights,
                            image_size=config.classifier.image_size,
                            label_smoothing=config.classifier.label_smoothing).to(device)
print('done')

/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  33%|███▎      | 1/3 [00:00<00:00,  2.76it/s]An error occurred while trying to fetch /home/scpark/.cache/huggingface/hub/models--facebook--DiT-XL-2-256/snapshots/eab8

done


In [5]:
# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.taylor.solver.gdual_solver import GDual_Solver
from solvers.taylor.transform.logaffine_transform import LogAffineTransform
from solvers.taylor.extractor.gap_extractor import Extractor

noise_schedule = model.get_noise_schedule()
extractor = Extractor()
transform = LogAffineTransform(gamma_push=True, gamma_max=2, tau_offset=1, kappa_max=2, eps=1e-2)
solver = GDual_Solver(
    noise_schedule,
    steps=5,
    transform=transform,
    param_extractor=extractor,
    skip_type="time_uniform",
    pred_order=1,
    corr_order=2,
    order1_kappa=True,
    order2_kappa=True,
    use_corrector=True,
    time_learning=True,
    train_mode=True
).to(device)

optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)
print('solver/optimizer')

solver/optimizer


In [6]:
# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset

if config.main_loss != 'classifier':
    train_dataset = PtDataset(config.train_pt_dir)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    print('len(train_dataset) :', len(train_dataset))
    
valid_dataset = PtDataset(config.valid_pt_dir)
print('len(valid_dataset) :', len(valid_dataset))

valid_loader = DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')


len(valid_dataset) : 0
dataloaders ready


In [7]:
# ===============================
# Utils
# ===============================

def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True)
        raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, optimizer):
    ckpt = {
        "global_step": int(global_step),
        "optim_state_dict": optimizer.state_dict(),
        "solver_state_dict": solver.state_dict(),
        "config": dict(config),
    }
    os.makedirs(save_dir, exist_ok=True)
    step_path = os.path.join(save_dir, f"step_{global_step:08d}.pt")
    torch.save(ckpt, step_path)
    return step_path

In [8]:
@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    losses = {}
    if 'PSNR' in config.losses:
        losses['PSNR'] = []
    if 'inception' in config.losses:
        losses['inception'] = []
    if 'classifier' in config.losses:
        losses['classifier'] = []
    
    for batch in valid_loader:
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)
        target_features= batch['inception_feature'][:, 0].to(device, non_blocking=True)
        
        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.no_grad():
            latent_pred = solver.sample(noises, model_fn)
            if 'PSNR' in losses:
                psnr_loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                losses['PSNR'].append(psnr_loss.item())

            if 'inception' in config.losses or 'classifier' in config.losses:
                sample_pred = model.decode_vae(latent_pred, raw_output=True)
    
                if 'inception' in config.losses:
                    pred = inception(sample_pred)
                    inception_loss = F.mse_loss(pred, target_features)
                    losses['inception'].append(inception_loss.item())

                if 'classifier' in config.losses:
                    class_ids = conds.to(device, non_blocking=True).long()
                    ce_loss = classifier(sample_pred, targets=class_ids)["loss"]
                    losses['classifier'].append(ce_loss.item())

    for key in losses:
        losses[key] = float(np.mean(losses[key]))
    return losses

In [9]:
from IPython.display import clear_output

def do_train_loop(device, writer, solver, optimizer, global_step):
    solver.train()
    if config.main_loss == 'classifier':
        pbar = tqdm(range(100))
    else:
        pbar = tqdm(train_loader)
        
    for _, batch in enumerate(pbar):
        if global_step >= config.total_steps:
            break

        #if global_step > 0 and global_step % config.val_every == 0:
        if global_step % config.val_every == 0:
            valid_losses = get_valid_loss(device, solver)
            for key in valid_losses:
                writer.add_scalar(key, valid_losses[key], global_step)
            save_checkpoint(global_step, config.log_dir, solver, optimizer)

        optimizer.zero_grad(set_to_none=True)
        if config.main_loss == 'classifier':
            noises = torch.randn(config.batch_size, *config.latent_size).to(device, non_blocking=True)
            conds = torch.randint(0, 1000, size=(len(noises),))
        else:
            noises = batch['noise'].to(device, non_blocking=True)
            conds  = batch['cond']
            targets= batch['sample'].to(device, non_blocking=True)
            target_features = batch['inception_feature'][:, 0].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            latent_pred = solver.sample(noises, model_fn)
            if 'PSNR' == config.main_loss:
                loss = torch.log(F.mse_loss(latent_pred, targets) + 1e-8)
                
            if 'inception' == config.main_loss or 'classifier' == config.main_loss:
                sample_pred = model.decode_vae(latent_pred, raw_output=True)
    
                if 'inception' == config.main_loss:
                    pred = inception(sample_pred)
                    loss = F.mse_loss(pred, target_features)
                    
                if 'classifier' == config.main_loss:
                    class_ids = conds.to(device, non_blocking=True).long()
                    loss = classifier(sample_pred, targets=class_ids)["loss"]
                    
        abort_if_bad("train", loss, global_step)  # ← 즉시 중단

        loss.backward()
        
        # ---- 2) grad norm 기준 클리핑
        torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        optimizer.step()
        
        lr_now = optimizer.param_groups[0]["lr"]
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now})
        global_step += 1
        clear_output()

    return global_step



In [ ]:
# ===============================
# Train
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    while True:
        if global_step >= config.total_steps:
            break
        global_step = do_train_loop(device, writer, solver, optimizer, global_step)
    print('E-N-D')
    
if __name__ == "__main__":
    main()


  7%|▋         | 7/100 [00:12<02:44,  1.77s/it, loss=0.248, lr=0.002]